In [1]:
import pandas as pd
import os

portfolio_config = {
    'data/holdings/CNDX.csv': 0.15,      # NASDAQ 100
    'data/holdings/IGLN.csv': 0.05,      # Złoto
    'data/holdings/mWIG40TR.csv': 0.04,  # mWIG40
    'data/holdings/URNU.csv': 0.05,      # Uran
    'data/holdings/VFEA.csv': 0.10,      # Rynki wschodzące
    'data/holdings/VVSM.csv': 0.05,      # Półprzewodniki
    'data/holdings/WIG20TR.csv': 0.06,   # WIG20
    'data/holdings/WEBN.csv': 0.40,      # Globalny
    'data/holdings/WSML.csv': 0.10,      # Małe spółki
      
}

dataframes = []

# 2. Pętla wczytująca pliki
for file_path, portfolio_weight in portfolio_config.items():
    try:
        df = pd.read_csv(file_path)
        
        # --- Ekstrakcja nazwy ETF ze ścieżki ---
        # os.path.basename('data/cspx.csv') zwróci samo 'cspx.csv'
        # .replace('.csv', '') wytnie rozszerzenie, a .upper() powiększy litery -> 'CSPX'
        etf_name = os.path.basename(file_path).replace('.csv', '').upper()
        df['Source_ETF'] = etf_name
        
        # Czyszczenie danych (wielkość liter i puste wartości)
        if 'Name' in df.columns:
            df['Name'] = df['Name'].astype(str).str.upper()
            
        cols_to_fill = ['Sector', 'Country', 'Currency']
        for col in cols_to_fill:
            if col in df.columns:
                df[col] = df[col].fillna('UNKNOWN')
                
        # Obliczenie efektywnej wagi
        df['Effective_Weight'] = df['Weight'] * portfolio_weight
        
        dataframes.append(df)
        
    except FileNotFoundError:
        print(f"Błąd: Nie znaleziono pliku {file_path}. Upewnij się, że folder 'data' istnieje.")

# 3. Połączenie wszystkich danych
df_holdings = pd.concat(dataframes, ignore_index=True)

# 4. TOP 100 Pozycji z łączeniem nazw ETF
# Używamy słownika agregacji: dla wagi robimy sumę, dla ETF łączymy unikalne wartości
top_100_companies = df_holdings.groupby(['Name']).agg({ #TODO: 'ISIN code', 
    'Effective_Weight': 'sum',
    'Source_ETF': lambda x: ', '.join(sorted(x.unique()))
}).reset_index()

# Sortowanie i przycięcie do TOP 100
top_100_companies = top_100_companies.sort_values(by='Effective_Weight', ascending=False).head(100)

# 5. Pozostałe agregacje (Sektor, Kraj, Waluta)
sector_alloc = df_holdings.groupby('Sector')['Effective_Weight'].sum().reset_index().sort_values(by='Effective_Weight', ascending=False)
country_alloc = df_holdings.groupby('Country')['Effective_Weight'].sum().reset_index().sort_values(by='Effective_Weight', ascending=False)
currency_alloc = df_holdings.groupby('Currency')['Effective_Weight'].sum().reset_index().sort_values(by='Effective_Weight', ascending=False)

# 6. Funkcja formatująca wyniki do wyświetlenia
def format_to_percentages(df_to_format):
    df_copy = df_to_format.copy()
    df_copy['Allocation_%'] = (df_copy['Effective_Weight'] * 100).round(2).astype(str) + '%'
    return df_copy.drop(columns=['Effective_Weight']).reset_index(drop=True)

# Wyświetlanie wyników
print("🏆 TOP 100 POZYCJI W PORTFELU (W TYM ŹRÓDŁO ETF)")
display(format_to_percentages(top_100_companies))

print("\n🏭 ALOKACJA SEKTOROWA")
display(format_to_percentages(sector_alloc))

print("\n🌍 ALOKACJA GEOGRAFICZNA (KRAJ)")
display(format_to_percentages(country_alloc))

print("\n💵 ALOKACJA WALUTOWA")
display(format_to_percentages(currency_alloc))

🏆 TOP 100 POZYCJI W PORTFELU (W TYM ŹRÓDŁO ETF)


,Name,Source_ETF,Allocation_%
0,GOLD,IGLN,5.0%
1,NVIDIA CORP,"CNDX, VVSM, WEBN",3.7%
2,APPLE INC,"CNDX, WEBN",2.64%
3,MICROSOFT CORP,"CNDX, WEBN",2.14%
4,BROADCOM INC,"CNDX, VVSM, WEBN",1.81%
...,...,...,...
95,CHEVRON CORP,WEBN,0.14%
96,DEVELIA SA,MWIG40TR,0.14%
97,HOME DEPOT INC,WEBN,0.14%
98,BANK OF AMERICA CORP,WEBN,0.14%



🏭 ALOKACJA SEKTOROWA


,Sector,Allocation_%
0,Information Technology,28.85%
1,Financials,11.62%
2,Industrials,9.3%
3,Consumer Discretionary,8.5%
4,Communication Services,6.85%
5,Energy,6.55%
6,Health Care,5.39%
7,Commodities,5.0%
8,Consumer Staples,3.92%
9,Materials,3.61%



🌍 ALOKACJA GEOGRAFICZNA (KRAJ)


,Country,Allocation_%
0,United States,50.63%
1,Poland,10.03%
2,Global,5.0%
3,Taiwan,4.12%
4,Canada,3.7%
5,Japan,3.67%
6,Hong Kong,2.56%
7,India,2.19%
8,China,1.93%
9,United Kingdom,1.76%



💵 ALOKACJA WALUTOWA


,Currency,Allocation_%
0,USD,57.1%
1,PLN,10.03%
2,EUR,4.06%
3,JPY,3.67%
4,TWD,3.66%
5,CAD,3.57%
6,HKD,3.56%
7,INR,2.19%
8,GBP,1.8%
9,KRW,1.7%


In [2]:
from pathlib import Path
from datetime import datetime
import html as html_lib

# --- Report settings ---
output_path = Path('data/holdings_report.html')
report_title = 'ETF Allocation Intelligence Report'
default_chart_mode = 'bar'  # Default view after opening HTML: 'bar' or 'pie'
author_name = 'azalurg'
github_url = 'https://github.com/Azalurg/stock-market-research'


def add_share_column(df, value_col='Effective_Weight', total_weight=None):
    tmp = df.copy()
    if total_weight is None:
        total_weight = tmp[value_col].sum()
    tmp['Share_%'] = (tmp[value_col] / total_weight * 100).round(2)
    return tmp


def build_bar_rows(df, label_col, value_col='Share_%', max_rows=15):
    rows = []
    for _, row in df.head(max_rows).iterrows():
        label = html_lib.escape(str(row[label_col]))
        pct = float(row[value_col])
        bar_width = max(0.0, min(pct, 100.0))
        rows.append(
            f"""
            <div class='bar-row'>
                <div class='bar-head'>
                    <span class='bar-label'>{label}</span>
                    <span class='bar-value'>{pct:.2f}%</span>
                </div>
                <div class='bar-track'>
                    <div class='bar-fill' style='width:{bar_width:.2f}%;'></div>
                </div>
            </div>
            """
        )
    return '\n'.join(rows)


def build_pie_chart(df, label_col, value_col='Share_%', max_slices=10):
    colors = [
        '#1b2a41', '#2c4f7c', '#44699c', '#6f86b8', '#8aa1c7',
        '#b9925a', '#8a6f45', '#495057', '#6c757d', '#343a40',
        '#264653', '#3d5a80'
    ]

    view = df.head(max_slices).copy()
    view[value_col] = view[value_col].astype(float)
    covered = float(view[value_col].sum())
    other_pct = max(0.0, round(100.0 - covered, 2))

    slices = []
    for _, row in view.iterrows():
        slices.append((html_lib.escape(str(row[label_col])), float(row[value_col])))

    if other_pct > 0.01:
        slices.append(('OTHERS', other_pct))

    gradient_parts = []
    legend_parts = []
    cumulative = 0.0

    for i, (label, pct) in enumerate(slices):
        color = colors[i % len(colors)]
        start = cumulative
        end = cumulative + pct
        gradient_parts.append(f"{color} {start:.2f}% {end:.2f}%")
        legend_parts.append(
            f"""
            <div class='legend-item'>
              <span class='legend-dot' style='background:{color};'></span>
              <span class='legend-label'>{label}</span>
              <span class='legend-value'>{pct:.2f}%</span>
            </div>
            """
        )
        cumulative = end

    gradient_css = ', '.join(gradient_parts) if gradient_parts else '#d7d7d7 0% 100%'

    return f"""
    <div class='pie-layout'>
      <div class='pie-chart' style='background: conic-gradient({gradient_css});'></div>
      <div class='pie-legend'>
        {''.join(legend_parts)}
      </div>
    </div>
    """


def build_dual_chart_block(df, label_col, value_col='Share_%', max_rows=12):
    bar_html = build_bar_rows(df, label_col, value_col=value_col, max_rows=max_rows)
    pie_html = build_pie_chart(df, label_col, value_col=value_col, max_slices=max_rows)

    if default_chart_mode.lower() == 'pie':
        bar_class = 'chart-view hidden'
        pie_class = 'chart-view'
    else:
        bar_class = 'chart-view'
        pie_class = 'chart-view hidden'

    return f"""
    <div class='chart-block'>
      <div class='{bar_class}' data-mode='bar'>
        {bar_html}
      </div>
      <div class='{pie_class}' data-mode='pie'>
        {pie_html}
      </div>
    </div>
    """


# -------------------------------------------------
# Important: shares are calculated from the full portfolio, not TOP 100
# -------------------------------------------------
portfolio_total_weight = df_holdings['Effective_Weight'].sum()

all_companies = (
    df_holdings.groupby('Name')
    .agg({
        'Effective_Weight': 'sum',
        'Source_ETF': lambda x: ', '.join(sorted(x.unique()))
    })
    .reset_index()
    .sort_values(by='Effective_Weight', ascending=False)
)

company_share = add_share_column(all_companies, 'Effective_Weight', total_weight=portfolio_total_weight)
sector_share = add_share_column(sector_alloc, 'Effective_Weight', total_weight=portfolio_total_weight)
country_share = add_share_column(country_alloc, 'Effective_Weight', total_weight=portfolio_total_weight)
currency_share = add_share_column(currency_alloc, 'Effective_Weight', total_weight=portfolio_total_weight)

# Views used in report
top_20_companies_view = company_share.head(20).copy()

# KPI
num_holdings = len(company_share)
num_sectors = sector_share['Sector'].nunique()
num_countries = country_share['Country'].nunique()
num_currencies = currency_share['Currency'].nunique()

largest_position = company_share.iloc[0]
largest_sector = sector_share.iloc[0]
largest_country = country_share.iloc[0]
largest_currency = currency_share.iloc[0]

top_10_share = company_share['Share_%'].head(10).sum()
top_3_sector_share = sector_share['Share_%'].head(3).sum()
median_position_share = company_share['Share_%'].median()
effective_n_holdings = (100.0 ** 2) / ((company_share['Share_%'] ** 2).sum())

# TOP 20 table
top20 = top_20_companies_view[['Name', 'Source_ETF', 'Share_%']].copy()
top20['Share_%'] = top20['Share_%'].map(lambda x: f"{x:.2f}%")
top20_html = top20.to_html(index=False, classes='table table-top', border=0)

# Charts (both versions included, switched by JS in HTML)
sector_chart_html = build_dual_chart_block(sector_share, 'Sector', max_rows=12)
country_chart_html = build_dual_chart_block(country_share, 'Country', max_rows=12)
currency_chart_html = build_dual_chart_block(currency_share, 'Currency', max_rows=12)

updated_at = datetime.now().strftime('%Y-%m-%d %H:%M')

bar_active = 'active' if default_chart_mode == 'bar' else ''
pie_active = 'active' if default_chart_mode == 'pie' else ''

html = f"""
<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='UTF-8' />
  <meta name='viewport' content='width=device-width, initial-scale=1.0' />
  <title>{report_title}</title>
  <style>
    :root {{
      --bg: #eef2f7;
      --surface: #ffffff;
      --surface-2: #f7f9fc;
      --text: #1e293b;
      --muted: #64748b;
      --line: #d5deea;
      --navy: #1b2a41;
      --navy-2: #2c4f7c;
      --steel: #6f86b8;
      --gold: #b9925a;
      --ok: #0f766e;
    }}

    * {{ box-sizing: border-box; }}

    body {{
      margin: 0;
      padding: 24px;
      font-family: 'IBM Plex Sans', 'Public Sans', 'Segoe UI', sans-serif;
      color: var(--text);
      background:
        linear-gradient(160deg, #f5f8fc 0%, #ebf1f8 50%, #e7eef6 100%);
    }}

    .container {{
      max-width: 1320px;
      margin: 0 auto;
      display: grid;
      gap: 16px;
    }}

    .header {{
      background: linear-gradient(135deg, #162338 0%, #20344f 70%, #29466b 100%);
      color: #ffffff;
      border-radius: 16px;
      padding: 22px;
      box-shadow: 0 14px 36px rgba(27, 42, 65, 0.28);
      display: grid;
      gap: 10px;
      border: 1px solid rgba(255, 255, 255, 0.14);
    }}

    .title-row {{
      display: flex;
      justify-content: space-between;
      align-items: flex-start;
      gap: 12px;
      flex-wrap: wrap;
    }}

    .header h1 {{
      margin: 0;
      font-size: 30px;
      line-height: 1.1;
      letter-spacing: 0.2px;
      font-family: 'IBM Plex Serif', 'Georgia', serif;
      font-weight: 600;
    }}

    .subtitle {{
      margin: 6px 0 0;
      color: rgba(255, 255, 255, 0.82);
      font-size: 14px;
    }}

    .meta-badge {{
      border: 1px solid rgba(255, 255, 255, 0.25);
      border-radius: 999px;
      padding: 6px 12px;
      font-size: 12px;
      color: rgba(255, 255, 255, 0.92);
      background: rgba(255, 255, 255, 0.08);
    }}

    .mode-toggle {{
      display: inline-flex;
      width: fit-content;
      border: 1px solid rgba(255, 255, 255, 0.28);
      border-radius: 999px;
      background: rgba(255, 255, 255, 0.08);
      overflow: hidden;
    }}

    .mode-btn {{
      border: 0;
      background: transparent;
      color: rgba(255, 255, 255, 0.78);
      padding: 8px 14px;
      cursor: pointer;
      font-weight: 600;
      font-size: 13px;
    }}

    .mode-btn.active {{
      background: #ffffff;
      color: var(--navy);
    }}

    .kpi-grid {{
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(190px, 1fr));
      gap: 12px;
    }}

    .kpi {{
      background: var(--surface);
      border: 1px solid var(--line);
      border-radius: 14px;
      padding: 16px;
      box-shadow: 0 10px 22px rgba(27, 42, 65, 0.06);
    }}

    .kpi .label {{
      color: var(--muted);
      font-size: 11px;
      text-transform: uppercase;
      letter-spacing: 0.9px;
      font-weight: 700;
    }}

    .kpi .value {{
      margin-top: 8px;
      font-size: 28px;
      font-weight: 700;
      color: var(--navy);
    }}

    .kpi .sub {{
      margin-top: 6px;
      font-size: 13px;
      color: var(--muted);
    }}

    .grid-2 {{
      display: grid;
      grid-template-columns: 1.3fr 1fr;
      gap: 16px;
    }}

    .card {{
      background: var(--surface);
      border: 1px solid var(--line);
      border-radius: 16px;
      padding: 16px;
      box-shadow: 0 8px 20px rgba(27, 42, 65, 0.05);
    }}

    .card h2 {{
      margin: 0 0 12px;
      font-size: 18px;
      color: var(--navy);
      font-family: 'IBM Plex Serif', 'Georgia', serif;
      font-weight: 600;
    }}

    .table {{
      width: 100%;
      border-collapse: collapse;
      font-size: 14px;
    }}

    .table th, .table td {{
      text-align: left;
      padding: 10px 8px;
      border-bottom: 1px solid #e3eaf3;
      vertical-align: top;
    }}

    .table th {{
      color: var(--muted);
      font-weight: 700;
      font-size: 11px;
      text-transform: uppercase;
      letter-spacing: 0.8px;
      background: var(--surface-2);
    }}

    .hidden {{ display: none; }}

    .bar-row + .bar-row {{ margin-top: 10px; }}
    .bar-head {{ display: flex; justify-content: space-between; gap: 8px; margin-bottom: 6px; font-size: 13px; }}
    .bar-label {{ color: var(--text); font-weight: 500; }}
    .bar-value {{ color: var(--muted); font-weight: 600; }}

    .bar-track {{
      width: 100%;
      height: 10px;
      border-radius: 999px;
      background: #e3eaf3;
      overflow: hidden;
    }}

    .bar-fill {{
      height: 100%;
      border-radius: 999px;
      background: linear-gradient(90deg, var(--navy), var(--steel));
    }}

    .pie-layout {{
      display: grid;
      grid-template-columns: 200px 1fr;
      gap: 14px;
      align-items: start;
    }}

    .pie-chart {{
      width: 200px;
      height: 200px;
      border-radius: 50%;
      border: 1px solid var(--line);
      box-shadow: inset 0 0 0 1px rgba(255, 255, 255, 0.6);
    }}

    .pie-legend {{ display: grid; gap: 6px; }}

    .legend-item {{
      display: grid;
      grid-template-columns: 12px 1fr auto;
      gap: 8px;
      align-items: center;
      font-size: 13px;
    }}

    .legend-dot {{
      width: 12px;
      height: 12px;
      border-radius: 50%;
      border: 1px solid rgba(0, 0, 0, 0.08);
    }}

    .legend-label {{ color: var(--text); }}
    .legend-value {{ color: var(--muted); font-weight: 600; }}

    .footer {{
      color: var(--muted);
      font-size: 12px;
      padding: 2px 2px 0;
      display: grid;
      gap: 4px;
      text-align: right;
    }}

    .footer-meta a {{
      color: var(--navy-2);
      text-decoration: none;
      font-weight: 600;
    }}

    .footer-meta a:hover {{
      text-decoration: underline;
    }}

    @media (max-width: 980px) {{
      .grid-2 {{ grid-template-columns: 1fr; }}
      .pie-layout {{ grid-template-columns: 1fr; }}
      .pie-chart {{ margin: 0 auto; }}
      .title-row {{ flex-direction: column; align-items: flex-start; }}
      body {{ padding: 14px; }}
    }}
  </style>
</head>
<body>
  <div class='container'>
    <section class='header'>
      <div class='title-row'>
        <div>
          <h1>{report_title}</h1>
          <p class='subtitle'>Institutional-style snapshot of equity, sector, geography and currency exposure.</p>
        </div>
        <div class='meta-badge'>Updated: {updated_at}</div>
      </div>
      <div class='mode-toggle' id='modeToggle'>
        <button type='button' class='mode-btn {bar_active}' data-mode='bar'>Bars</button>
        <button type='button' class='mode-btn {pie_active}' data-mode='pie'>Pie</button>
      </div>
    </section>

    <section class='kpi-grid'>
      <div class='kpi'>
        <div class='label'>Total Holdings</div>
        <div class='value'>{num_holdings}</div>
        <div class='sub'>Unique companies in portfolio</div>
      </div>
      <div class='kpi'>
        <div class='label'>Sectors</div>
        <div class='value'>{num_sectors}</div>
        <div class='sub'>Distinct sector buckets</div>
      </div>
      <div class='kpi'>
        <div class='label'>Countries</div>
        <div class='value'>{num_countries}</div>
        <div class='sub'>Geographic diversification</div>
      </div>
      <div class='kpi'>
        <div class='label'>Currencies</div>
        <div class='value'>{num_currencies}</div>
        <div class='sub'>FX exposure profile</div>
      </div>
      <div class='kpi'>
        <div class='label'>Largest Position</div>
        <div class='value'>{largest_position['Share_%']:.2f}%</div>
        <div class='sub'>{largest_position['Name']}</div>
      </div>
      <div class='kpi'>
        <div class='label'>Largest Sector</div>
        <div class='value'>{largest_sector['Share_%']:.2f}%</div>
        <div class='sub'>{largest_sector['Sector']}</div>
      </div>
      <div class='kpi'>
        <div class='label'>Largest Country</div>
        <div class='value'>{largest_country['Share_%']:.2f}%</div>
        <div class='sub'>{largest_country['Country']}</div>
      </div>
      <div class='kpi'>
        <div class='label'>Largest Currency</div>
        <div class='value'>{largest_currency['Share_%']:.2f}%</div>
        <div class='sub'>{largest_currency['Currency']}</div>
      </div>
      <div class='kpi'>
        <div class='label'>Top 10 Concentration</div>
        <div class='value'>{top_10_share:.2f}%</div>
        <div class='sub'>Combined weight of top 10 holdings</div>
      </div>
      <div class='kpi'>
        <div class='label'>Top 3 Sectors</div>
        <div class='value'>{top_3_sector_share:.2f}%</div>
        <div class='sub'>Share of three largest sectors</div>
      </div>
      <div class='kpi'>
        <div class='label'>Median Position</div>
        <div class='value'>{median_position_share:.3f}%</div>
        <div class='sub'>Median company weight</div>
      </div>
      <div class='kpi'>
        <div class='label'>Effective Holdings</div>
        <div class='value'>{effective_n_holdings:.0f}</div>
        <div class='sub'>Herfindahl-based diversification</div>
      </div>
    </section>

    <section class='grid-2'>
      <article class='card'>
        <h2>Top 20 Portfolio Positions</h2>
        {top20_html}
      </article>
      <article class='card'>
        <h2>Sector Allocation</h2>
        {sector_chart_html}
      </article>
    </section>

    <section class='grid-2'>
      <article class='card'>
        <h2>Geographic Allocation</h2>
        {country_chart_html}
      </article>
      <article class='card'>
        <h2>Currency Allocation</h2>
        {currency_chart_html}
      </article>
    </section>

    <div class='footer'>
      <div class='footer-meta'>
        Author: {author_name} | GitHub: <a href='{github_url}' target='_blank' rel='noopener noreferrer'>repository</a>
      </div>
      <div class='footer-meta'>
        Hobbyist document only. This material is not professional and should not be treated as investment advice.
      </div>
    </div>
  </div>

  <script>
    (function () {{
      const toggle = document.getElementById('modeToggle');
      if (!toggle) return;

      const buttons = Array.from(toggle.querySelectorAll('.mode-btn'));
      const chartViews = Array.from(document.querySelectorAll('.chart-view'));

      function applyMode(mode) {{
        buttons.forEach((btn) => {{
          const isActive = btn.dataset.mode === mode;
          btn.classList.toggle('active', isActive);
        }});

        chartViews.forEach((view) => {{
          const matches = view.dataset.mode === mode;
          view.classList.toggle('hidden', !matches);
        }});
      }}

      buttons.forEach((btn) => {{
        btn.addEventListener('click', () => applyMode(btn.dataset.mode));
      }});

      applyMode('{default_chart_mode}');
    }})();
  </script>
</body>
</html>
"""

output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(html, encoding='utf-8')

print(f'Report saved to: {output_path.resolve()}')
print(f'Total unique companies: {num_holdings}')
print('You can switch chart type inside HTML with the Bars/Pie toggle.')

Report saved to: /home/azalurg/Github/gpw/data/holdings_report.html
Total unique companies: 8738
You can switch chart type inside HTML with the Bars/Pie toggle.
